# PREP

In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
from pyspark.sql import Window

SLV = "cms_partd_silver.core"
GLD = "cms_partd_gold.mart"

# Convenience
def tbl(name): 
    return spark.table(name)

def save_delta(df, table_name, mode="overwrite"):
    df.write.format("delta").mode(mode).saveAsTable(table_name)

# =========================
# 0) LOAD SILVER TABLES
# =========================
slv_plan = tbl(f"{SLV}.slv_plan")
slv_geo  = tbl(f"{SLV}.slv_geo")
slv_cost = tbl(f"{SLV}.slv_beneficiary_cost")
slv_prc  = tbl(f"{SLV}.slv_pricing")
slv_net  = tbl(f"{SLV}.slv_pharmacy_network")
slv_for  = tbl(f"{SLV}.slv_basic_formulary")



# 1. DIMENSIONS

## 1.1 dim_plan

In [0]:

# =========================
# 1) DIMENSIONS
# =========================

# 1.1 dim_plan
dim_plan_cols = [
    "plan_key","plan_bk","contract_id","plan_id","segment_id","contract_type",
    "contract_name","plan_name","formulary_id",
    "premium","deductible","snp",
    "ma_region_code","pdp_region_code","state","county_code",
    "plan_suppressed_yn"
]
dim_plan = slv_plan.select(*[c for c in dim_plan_cols if c in slv_plan.columns])

# Ensure uniqueness (plan_key grain)
dim_plan = dim_plan.dropDuplicates(["plan_key"])
save_delta(dim_plan, f"{GLD}.dim_plan")


## 1.2 dim_geography

In [0]:

# 1.2 dim_geography
dim_geo_cols = ["county_code","statename","county","ma_region_code","ma_region","pdp_region_code","pdp_region"]
dim_geography = slv_geo.select(*[c for c in dim_geo_cols if c in slv_geo.columns]).dropDuplicates(["county_code"])
save_delta(dim_geography, f"{GLD}.dim_geography")



## 1.3 dim_drug

In [0]:
# 1.3 dim_drug (NDC-focused, keep RXCUI if present)
# Pricing has NDC; formulary has NDC+RXCUI
dim_drug_from_prc = slv_prc.select("ndc").dropDuplicates(["ndc"])
dim_drug_from_for = (
    slv_for.select("ndc","rxcui")
    .where(F.col("ndc").isNotNull())
    .dropDuplicates(["ndc","rxcui"])
)

# If multiple RXCUI per NDC, keep the most frequent pair as "primary" (optional)
# We'll do a simple first() after grouping
dim_drug = (
    dim_drug_from_for.groupBy("ndc")
    .agg(F.first("rxcui", ignorenulls=True).alias("rxcui"))
    .unionByName(dim_drug_from_prc.withColumn("rxcui", F.lit(None).cast("string")))
    .groupBy("ndc").agg(F.first("rxcui", ignorenulls=True).alias("rxcui"))
)
save_delta(dim_drug, f"{GLD}.dim_drug")



## 1.4 dim_days_supply

In [0]:
# 1.4 dim_days_supply (combine coded + numeric)
# beneficiary_cost: days_supply is coded (1/2/3/4) + label
days_supply_coded = (
    slv_cost.select(
        F.col("days_supply").cast("int").alias("days_supply_code"),
        F.col("days_supply_label").cast("string").alias("days_supply_value")
    )
    .dropDuplicates()
)

# pricing: days_supply is numeric (30/60/90)
days_supply_numeric = (
    slv_prc.select(F.col("days_supply").cast("int").alias("days_supply_value"))
    .dropDuplicates()
    .withColumn("days_supply_code", F.lit(None).cast("int"))
)

dim_days_supply = (
    days_supply_coded.unionByName(days_supply_numeric.select(days_supply_coded.columns), allowMissingColumns=True)
    .dropDuplicates(["days_supply_code","days_supply_value"])
)
save_delta(dim_days_supply, f"{GLD}.dim_days_supply")



## 1.5 dim_tier

In [0]:
# 1.5 dim_tier
dim_tier = (
    slv_cost.select(F.col("tier").cast("int").alias("tier"))
    .where(F.col("tier").isNotNull())
    .dropDuplicates(["tier"])
    .orderBy("tier")
)
save_delta(dim_tier, f"{GLD}.dim_tier")



## 1.6 dim_coverage_level

In [0]:
# 1.6 dim_coverage_level
dim_coverage_level = (
    slv_cost.select(F.col("coverage_level").cast("int").alias("coverage_level"))
    .where(F.col("coverage_level").isNotNull())
    .dropDuplicates(["coverage_level"])
    .orderBy("coverage_level")
)
save_delta(dim_coverage_level, f"{GLD}.dim_coverage_level")




## 1.7 dim_pharmacy

In [0]:
# 1.7 dim_pharmacy
dim_pharmacy_cols = ["pharmacy_number","pharmacy_zipcode","pharmacy_retail","pharmacy_mail"]
dim_pharmacy = slv_net.select(*[c for c in dim_pharmacy_cols if c in slv_net.columns]).dropDuplicates(["pharmacy_number"])
save_delta(dim_pharmacy, f"{GLD}.dim_pharmacy")

# 2. FACTS

## 2.1 fact_beneficiary_cost

In [0]:
# =========================
# 2) FACTS
# =========================

# 2.1 fact_beneficiary_cost
fact_beneficiary_cost_cols = [
    "cost_key","plan_key","coverage_level","tier","days_supply","days_supply_label",
    "cost_type_pref","cost_amt_pref","cost_min_amt_pref","cost_max_amt_pref","cost_type_pref_label",
    "cost_type_nonpref","cost_amt_nonpref","cost_min_amt_nonpref","cost_max_amt_nonpref","cost_type_nonpref_label",
    "cost_type_mail_pref","cost_amt_mail_pref","cost_min_amt_mail_pref","cost_max_amt_mail_pref","cost_type_mail_pref_label",
    "cost_type_mail_nonpref","cost_amt_mail_nonpref","cost_min_amt_mail_nonpref","cost_max_amt_mail_nonpref","cost_type_mail_nonpref_label",
    "tier_specialty_yn","ded_applies_yn",
    "ingest_ts","ingest_date","source_file"
]
fact_beneficiary_cost = slv_cost.select(*[c for c in fact_beneficiary_cost_cols if c in slv_cost.columns])
save_delta(fact_beneficiary_cost, f"{GLD}.fact_beneficiary_cost")



## 2.2 fact_pricing_unit_cost

In [0]:
# 2.2 fact_pricing_unit_cost
fact_pricing_cols = [
    "pricing_key","plan_key","ndc","days_supply","unit_cost",
    "ingest_ts","ingest_date","source_file"
]
fact_pricing_unit_cost = slv_prc.select(*[c for c in fact_pricing_cols if c in slv_prc.columns])
save_delta(fact_pricing_unit_cost, f"{GLD}.fact_pricing_unit_cost")

## 2.3 fact_pharmacy_network

In [0]:
# 2.3 fact_pharmacy_network
fact_network_cols = [
    "network_key","plan_key","pharmacy_number","pharmacy_zipcode",
    "preferred_status_retail","preferred_status_mail","pharmacy_retail","pharmacy_mail",
    "in_area_flag","is_in_area","floor_price",
    "brand_dispensing_fee_30","brand_dispensing_fee_60","brand_dispensing_fee_90",
    "generic_dispensing_fee_30","generic_dispensing_fee_60","generic_dispensing_fee_90",
    "avg_brand_fee","avg_generic_fee",
    "ingest_ts","ingest_date","source_file"
]
fact_pharmacy_network = slv_net.select(*[c for c in fact_network_cols if c in slv_net.columns])
save_delta(fact_pharmacy_network, f"{GLD}.fact_pharmacy_network")



## 2.4 fact_formulary_drug

In [0]:
# 2.4 fact_formulary_drug
fact_formulary_cols = [
    "formulary_drug_key","formulary_id","formulary_version","contract_year","rxcui","ndc",
    "tier_level_value",
    "quantity_limit_yn","prior_authorization_yn","step_therapy_yn",
    "quantity_limit_flag","prior_auth_flag","step_therapy_flag",
    "quantity_limit_amount","quantity_limit_days",
    "restriction_count",
    "ingest_ts","ingest_date","source_file"
]
fact_formulary_drug = slv_for.select(*[c for c in fact_formulary_cols if c in slv_for.columns])
save_delta(fact_formulary_drug, f"{GLD}.fact_formulary_drug")


# 3. AGGREGATES FOR POWER BI

## agg_plan_formulary_metrics (plan_key grain)

In [0]:

# =========================
# 3) AGGREGATES FOR POWER BI
#    agg_plan_formulary_metrics (plan_key grain)
# =========================
# Join dim_plan -> fact_formulary_drug by formulary_id
# This avoids many-to-many inside Power BI by materializing plan-level metrics.
plan_formulary = (
    dim_plan.select("plan_key","formulary_id")
    .join(fact_formulary_drug, on="formulary_id", how="left")
)

agg_plan_formulary_metrics = (
    plan_formulary.groupBy("plan_key")
    .agg(
        F.countDistinct("ndc").alias("formulary_breadth_ndc"),
        F.countDistinct("rxcui").alias("formulary_breadth_rxcui"),
        F.avg(F.col("tier_level_value").cast("double")).alias("avg_tier_level"),
        F.avg(F.coalesce(F.col("quantity_limit_flag").cast("double"), F.lit(0.0))).alias("ql_rate"),
        F.avg(F.coalesce(F.col("prior_auth_flag").cast("double"), F.lit(0.0))).alias("pa_rate"),
        F.avg(F.coalesce(F.col("step_therapy_flag").cast("double"), F.lit(0.0))).alias("st_rate"),
        F.avg(F.coalesce(F.col("restriction_count").cast("double"), F.lit(0.0))).alias("avg_restriction_count"),
        # Example restrictiveness index 0-100
        (F.lit(100.0) * (
            F.lit(0.4) * F.avg(F.coalesce(F.col("prior_auth_flag").cast("double"), F.lit(0.0))) +
            F.lit(0.3) * F.avg(F.coalesce(F.col("step_therapy_flag").cast("double"), F.lit(0.0))) +
            F.lit(0.2) * F.avg(F.coalesce(F.col("quantity_limit_flag").cast("double"), F.lit(0.0))) +
            F.lit(0.1) * F.when(F.count("*") > 0, F.lit(1.0)).otherwise(F.lit(0.0))
        )).alias("restrictiveness_index")
    )
)

save_delta(agg_plan_formulary_metrics, f"{GLD}.agg_plan_formulary_metrics")

## agg_plan_network_metrics

In [0]:

# Optional: other useful agg tables for BI performance
# 3.1 agg_plan_network_metrics
agg_plan_network_metrics = (
    fact_pharmacy_network.groupBy("plan_key")
    .agg(
        F.countDistinct("pharmacy_number").alias("network_size"),
        F.avg(F.col("preferred_status_retail").cast("double")).alias("preferred_retail_share"),
        F.avg(F.col("preferred_status_mail").cast("double")).alias("preferred_mail_share"),
        F.avg(F.col("is_in_area").cast("double")).alias("in_area_share"),
        F.avg(F.col("floor_price").cast("double")).alias("avg_floor_price"),
        F.avg(F.col("avg_brand_fee").cast("double")).alias("avg_brand_fee"),
        F.avg(F.col("avg_generic_fee").cast("double")).alias("avg_generic_fee"),
    )
)
save_delta(agg_plan_network_metrics, f"{GLD}.agg_plan_network_metrics")


## agg_plan_pricing_metrics

In [0]:

# 3.2 agg_plan_pricing_metrics
agg_plan_pricing_metrics = (
    fact_pricing_unit_cost.groupBy("plan_key")
    .agg(
        F.avg(F.col("unit_cost").cast("double")).alias("avg_unit_cost"),
        F.expr("percentile_approx(unit_cost, 0.9)").cast("double").alias("p90_unit_cost"),
        F.expr("percentile_approx(unit_cost, 0.1)").cast("double").alias("p10_unit_cost"),
        (F.expr("percentile_approx(unit_cost, 0.9)") - F.expr("percentile_approx(unit_cost, 0.1)")).cast("double").alias("p90_p10_spread_unit_cost"),
        F.countDistinct("ndc").alias("priced_ndc_count")
    )
)
save_delta(agg_plan_pricing_metrics, f"{GLD}.agg_plan_pricing_metrics")

## agg_plan_cost_structure_metrics

In [0]:
# agg_plan_cost_structure_metrics

# ----------------------------
# 1) Load base fact table
# ----------------------------
df = spark.table("cms_partd_gold.mart.fact_beneficiary_cost").select(
    "plan_key", "tier", "days_supply_label",
    "cost_type_pref", "cost_amt_pref",
    "cost_type_nonpref", "cost_amt_nonpref",
    "cost_type_mail_pref", "cost_amt_mail_pref",
    "cost_type_mail_nonpref", "cost_amt_mail_nonpref",
    "tier_specialty_yn", "ded_applies_yn"
)

# Cast all amount columns to double once
df = (
    df.withColumn("cost_amt_pref", F.col("cost_amt_pref").cast("double"))
      .withColumn("cost_amt_nonpref", F.col("cost_amt_nonpref").cast("double"))
      .withColumn("cost_amt_mail_pref", F.col("cost_amt_mail_pref").cast("double"))
      .withColumn("cost_amt_mail_nonpref", F.col("cost_amt_mail_nonpref").cast("double"))
)

# Helper: coinsurance rate (0-1) when type=2
pref_coins_rate     = F.when(F.col("cost_type_pref") == 2, F.col("cost_amt_pref")/100.0)
nonpref_coins_rate  = F.when(F.col("cost_type_nonpref") == 2, F.col("cost_amt_nonpref")/100.0)
mailpref_coins_rate = F.when(F.col("cost_type_mail_pref") == 2, F.col("cost_amt_mail_pref")/100.0)
mailnon_coins_rate  = F.when(F.col("cost_type_mail_nonpref") == 2, F.col("cost_amt_mail_nonpref")/100.0)

# Helper: copay amount when type=1
pref_copay_amt      = F.when(F.col("cost_type_pref") == 1, F.col("cost_amt_pref"))
nonpref_copay_amt   = F.when(F.col("cost_type_nonpref") == 1, F.col("cost_amt_nonpref"))
mailpref_copay_amt  = F.when(F.col("cost_type_mail_pref") == 1, F.col("cost_amt_mail_pref"))
mailnon_copay_amt   = F.when(F.col("cost_type_mail_nonpref") == 1, F.col("cost_amt_mail_nonpref"))

# ----------------------------
# 2) Plan-level metrics
# ----------------------------
plan_level = (
    df.groupBy("plan_key")
      .agg(
          # overall plan characteristics
          F.avg(F.when(F.col("ded_applies_yn") == "Y", 1.0).otherwise(0.0)).alias("ded_applies_rate"),
          F.avg(F.when(F.col("tier_specialty_yn") == "Y", 1.0).otherwise(0.0)).alias("specialty_row_share"),

          # ----------------- Preferred retail -----------------
          F.avg(F.when(F.col("cost_type_pref") == 0, 1.0).otherwise(0.0)).alias("pref_not_offered_share"),
          F.avg(F.when(F.col("cost_type_pref") == 1, 1.0).otherwise(0.0)).alias("pref_copay_share"),
          F.avg(F.when(F.col("cost_type_pref") == 2, 1.0).otherwise(0.0)).alias("pref_coinsurance_share"),

          F.avg(pref_copay_amt).alias("pref_avg_copay_amt"),
          F.percentile_approx(pref_copay_amt, 0.50).alias("pref_median_copay_amt"),
          F.percentile_approx(pref_copay_amt, 0.90).alias("pref_p90_copay_amt"),

          F.avg(pref_coins_rate).alias("pref_avg_coins_rate"),
          F.percentile_approx(pref_coins_rate, 0.50).alias("pref_median_coins_rate"),
          F.percentile_approx(pref_coins_rate, 0.90).alias("pref_p90_coins_rate"),

          # ----------------- Nonpreferred retail -----------------
          F.avg(F.when(F.col("cost_type_nonpref") == 0, 1.0).otherwise(0.0)).alias("nonpref_not_offered_share"),
          F.avg(F.when(F.col("cost_type_nonpref") == 1, 1.0).otherwise(0.0)).alias("nonpref_copay_share"),
          F.avg(F.when(F.col("cost_type_nonpref") == 2, 1.0).otherwise(0.0)).alias("nonpref_coinsurance_share"),

          F.avg(nonpref_copay_amt).alias("nonpref_avg_copay_amt"),
          F.percentile_approx(nonpref_copay_amt, 0.50).alias("nonpref_median_copay_amt"),
          F.percentile_approx(nonpref_copay_amt, 0.90).alias("nonpref_p90_copay_amt"),

          F.avg(nonpref_coins_rate).alias("nonpref_avg_coins_rate"),
          F.percentile_approx(nonpref_coins_rate, 0.50).alias("nonpref_median_coins_rate"),
          F.percentile_approx(nonpref_coins_rate, 0.90).alias("nonpref_p90_coins_rate"),

          # ----------------- Preferred mail -----------------
          F.avg(F.when(F.col("cost_type_mail_pref") == 0, 1.0).otherwise(0.0)).alias("mail_pref_not_offered_share"),
          F.avg(F.when(F.col("cost_type_mail_pref") == 1, 1.0).otherwise(0.0)).alias("mail_pref_copay_share"),
          F.avg(F.when(F.col("cost_type_mail_pref") == 2, 1.0).otherwise(0.0)).alias("mail_pref_coinsurance_share"),

          F.avg(mailpref_copay_amt).alias("mail_pref_avg_copay_amt"),
          F.percentile_approx(mailpref_copay_amt, 0.50).alias("mail_pref_median_copay_amt"),
          F.percentile_approx(mailpref_copay_amt, 0.90).alias("mail_pref_p90_copay_amt"),

          F.avg(mailpref_coins_rate).alias("mail_pref_avg_coins_rate"),
          F.percentile_approx(mailpref_coins_rate, 0.50).alias("mail_pref_median_coins_rate"),
          F.percentile_approx(mailpref_coins_rate, 0.90).alias("mail_pref_p90_coins_rate"),

          # ----------------- Nonpreferred mail -----------------
          F.avg(F.when(F.col("cost_type_mail_nonpref") == 0, 1.0).otherwise(0.0)).alias("mail_nonpref_not_offered_share"),
          F.avg(F.when(F.col("cost_type_mail_nonpref") == 1, 1.0).otherwise(0.0)).alias("mail_nonpref_copay_share"),
          F.avg(F.when(F.col("cost_type_mail_nonpref") == 2, 1.0).otherwise(0.0)).alias("mail_nonpref_coinsurance_share"),

          F.avg(mailnon_copay_amt).alias("mail_nonpref_avg_copay_amt"),
          F.percentile_approx(mailnon_copay_amt, 0.50).alias("mail_nonpref_median_copay_amt"),
          F.percentile_approx(mailnon_copay_amt, 0.90).alias("mail_nonpref_p90_copay_amt"),

          F.avg(mailnon_coins_rate).alias("mail_nonpref_avg_coins_rate"),
          F.percentile_approx(mailnon_coins_rate, 0.50).alias("mail_nonpref_median_coins_rate"),
          F.percentile_approx(mailnon_coins_rate, 0.90).alias("mail_nonpref_p90_coins_rate"),
      )
)

# ----------------------------
# 3) Tier-level preferred retail metrics (copay + coins rate)
# ----------------------------
pref_tier = (
    df.groupBy("plan_key")
      .agg(
          # COPAY ($) tier-level
          F.avg(F.when((F.col("tier")==1) & (F.col("cost_type_pref")==1), F.col("cost_amt_pref"))).alias("pref_avg_copay_tier_1"),
          F.avg(F.when((F.col("tier")==2) & (F.col("cost_type_pref")==1), F.col("cost_amt_pref"))).alias("pref_avg_copay_tier_2"),
          F.avg(F.when((F.col("tier")==3) & (F.col("cost_type_pref")==1), F.col("cost_amt_pref"))).alias("pref_avg_copay_tier_3"),
          F.avg(F.when((F.col("tier")==4) & (F.col("cost_type_pref")==1), F.col("cost_amt_pref"))).alias("pref_avg_copay_tier_4"),
          F.avg(F.when((F.col("tier")==5) & (F.col("cost_type_pref")==1), F.col("cost_amt_pref"))).alias("pref_avg_copay_tier_5"),
          F.avg(F.when((F.col("tier")==6) & (F.col("cost_type_pref")==1), F.col("cost_amt_pref"))).alias("pref_avg_copay_tier_6"),
          F.avg(F.when((F.col("tier")==7) & (F.col("cost_type_pref")==1), F.col("cost_amt_pref"))).alias("pref_avg_copay_tier_7"),

          # COINSURANCE rate (0-1) tier-level
          F.avg(F.when((F.col("tier")==1) & (F.col("cost_type_pref")==2), F.col("cost_amt_pref")/100.0)).alias("pref_avg_coins_rate_tier_1"),
          F.avg(F.when((F.col("tier")==2) & (F.col("cost_type_pref")==2), F.col("cost_amt_pref")/100.0)).alias("pref_avg_coins_rate_tier_2"),
          F.avg(F.when((F.col("tier")==3) & (F.col("cost_type_pref")==2), F.col("cost_amt_pref")/100.0)).alias("pref_avg_coins_rate_tier_3"),
          F.avg(F.when((F.col("tier")==4) & (F.col("cost_type_pref")==2), F.col("cost_amt_pref")/100.0)).alias("pref_avg_coins_rate_tier_4"),
          F.avg(F.when((F.col("tier")==5) & (F.col("cost_type_pref")==2), F.col("cost_amt_pref")/100.0)).alias("pref_avg_coins_rate_tier_5"),
          F.avg(F.when((F.col("tier")==6) & (F.col("cost_type_pref")==2), F.col("cost_amt_pref")/100.0)).alias("pref_avg_coins_rate_tier_6"),
          F.avg(F.when((F.col("tier")==7) & (F.col("cost_type_pref")==2), F.col("cost_amt_pref")/100.0)).alias("pref_avg_coins_rate_tier_7"),
      )
)

# ----------------------------
# 4) Final join + write Delta table
# ----------------------------
out = (
    plan_level.join(pref_tier, on="plan_key", how="left")
              .withColumn("agg_ts", F.current_timestamp())
)

(out.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable("cms_partd_gold.mart.agg_plan_cost_structure_metrics"))


## agg_plan_insulin_cost_metrics

In [0]:
plan_map = spark.table(f"{SLV}.slv_plan").select(
    "plan_key","formulary_id"
).dropDuplicates(["plan_key"])


In [0]:
# agg_plan_insulin_cost_metrics

# A plan-level summary: “Does this plan meet insulin affordability expectations?”
# It leverages your slv_plan_insulin_cost, which already contains per-plan insulin copay rules and an exceeds_cap_flag.

slv_ins_cost = tbl(f"{SLV}.slv_plan_insulin_cost")

# Canonicalize days supply buckets (adapt to your labels)
is_30 = F.lower(F.col("days_supply_label")).like("%30%")
is_90 = F.lower(F.col("days_supply_label")).like("%90%")

agg_plan_insulin_cost_metrics = (
    slv_ins_cost
    .groupBy("plan_key")
    .agg(
        F.min("insulin_min_copay_all_channels").alias("insulin_min_copay_plan"),
        F.max("insulin_max_copay_all_channels").alias("insulin_max_copay_plan"),
        F.avg(F.col("exceeds_cap_flag").cast("double")).alias("insulin_exceeds_cap_rate"),
        F.max(F.col("exceeds_cap_flag").cast("int")).alias("insulin_exceeds_cap_any_flag"),
        F.count(F.lit(1)).alias("insulin_cost_rows"),

        F.min(F.when(is_30, F.col("insulin_min_copay_all_channels"))).alias("insulin_min_copay_30d"),
        F.max(F.when(is_30, F.col("insulin_max_copay_all_channels"))).alias("insulin_max_copay_30d"),
        F.max(F.when(is_30, F.col("exceeds_cap_flag").cast("int"))).alias("insulin_exceeds_cap_30d_flag"),

        F.min(F.when(is_90, F.col("insulin_min_copay_all_channels"))).alias("insulin_min_copay_90d"),
        F.max(F.when(is_90, F.col("insulin_max_copay_all_channels"))).alias("insulin_max_copay_90d"),
        F.max(F.when(is_90, F.col("exceeds_cap_flag").cast("int"))).alias("insulin_exceeds_cap_90d_flag"),
    )
    .withColumn("gold_ts", F.current_timestamp())
)

save_delta(agg_plan_insulin_cost_metrics, f"{GLD}.agg_plan_insulin_cost_metrics")

## agg_plan_insulin_access_metrics

In [0]:
# agg_plan_insulin_access_metrics
# Plan-level insulin access barriers based on excluded/conditional coverage and PA/ST/QL.


form = spark.table(f"{SLV}.slv_basic_formulary")

ins_form = (
    plan_map.join(
        form,
        on=["formulary_id"],
        how="left"
    )
    .where(F.col("is_insulin")==1)
)



gold = (
  ins_form.groupBy("plan_key")
  .agg(
        F.countDistinct("ndc").alias("insulin_ndc_count"),
        F.avg(F.col("tier_level_value").cast("double")).alias("insulin_avg_tier_level"),
        F.avg(F.coalesce(F.col("quantity_limit_flag").cast("double"), F.lit(0.0))).alias("insulin_ql_rate"),
        F.avg(F.coalesce(F.col("prior_auth_flag").cast("double"), F.lit(0.0))).alias("insulin_pa_rate"),
        F.avg(F.coalesce(F.col("step_therapy_flag").cast("double"), F.lit(0.0))).alias("insulin_st_rate"),
        F.avg((F.col("prior_auth_flag").cast("int")+F.col("step_therapy_flag").cast("int")+F.col("quantity_limit_flag").cast("int")).cast("double")).alias("insulin_avg_restriction_count")
  )
  .withColumn(
      "insulin_restrictiveness_index",
      F.round(100*(0.5*F.col("insulin_pa_rate")+0.3*F.col("insulin_st_rate")+0.2*F.col("insulin_ql_rate")), 2)
  )
  .withColumn("gold_ts", F.current_timestamp())
)

gold.write.format("delta").mode("overwrite").saveAsTable("cms_partd_gold.mart.agg_plan_insulin_access_metrics")


## agg_plan_insulin_friendliness

In [0]:
# agg_plan_insulin_friendliness

cost = spark.table("cms_partd_gold.mart.agg_plan_insulin_cost_metrics").drop("gold_ts")
acc  = spark.table("cms_partd_gold.mart.agg_plan_insulin_access_metrics").drop("gold_ts")

df = cost.join(acc, on="plan_key", how="left").fillna({
    "insulin_pa_rate": 0.0, "insulin_st_rate": 0.0, "insulin_ql_rate": 0.0,
    "insulin_restrictiveness_index": 0.0, "insulin_ndc_count": 0
})

# Transparent rules
df = df.withColumn(
    "insulin_cost_risk_flag",
    F.when((F.col("insulin_exceeds_cap_any_flag")==1) | (F.col("insulin_exceeds_cap_rate")>0.05), 1).otherwise(0)
)

df = df.withColumn(
    "insulin_access_risk_flag",
    F.when((F.col("insulin_restrictiveness_index")>20) | (F.col("insulin_pa_rate")>0.20), 1).otherwise(0)
)

df = df.withColumn(
    "insulin_risk_flag",
    F.when((F.col("insulin_cost_risk_flag")==1) | (F.col("insulin_access_risk_flag")==1), 1).otherwise(0)
)

# Score (0-100): cost compliance + access friendliness
cost_score = F.greatest(F.lit(0.0), F.least(F.lit(100.0), (1.0 - F.col("insulin_exceeds_cap_rate"))*100.0))
access_score = F.greatest(F.lit(0.0), F.least(F.lit(100.0), (1.0 - F.col("insulin_restrictiveness_index")/100.0)*100.0))

df = (
  df.withColumn("insulin_cost_score", F.round(cost_score, 2))
    .withColumn("insulin_access_score", F.round(access_score, 2))
    .withColumn("insulin_friendliness_score", F.round(0.5*cost_score + 0.5*access_score, 2))
    .withColumn("gold_ts", F.current_timestamp())
)

df.write.format("delta").mode("overwrite").saveAsTable("cms_partd_gold.mart.agg_plan_insulin_friendliness")
# display(df)


In [0]:

# =========================
# 4) OPTIMIZE + ZORDER (suggestions)
# =========================
# Note: OPTIMIZE requires Databricks Runtime with Delta optimization support and permissions.
# Run these in a SQL cell if preferred; here we do spark.sql.

opt_statements = [
    # Dimensions (usually small; optimize optional)
    f"OPTIMIZE {GLD}.dim_plan ZORDER BY (plan_key)",
    f"OPTIMIZE {GLD}.dim_geography ZORDER BY (county_code)",
    f"OPTIMIZE {GLD}.dim_drug ZORDER BY (ndc)",
    f"OPTIMIZE {GLD}.dim_pharmacy ZORDER BY (pharmacy_number)",

    # Facts (important)
    f"OPTIMIZE {GLD}.fact_beneficiary_cost ZORDER BY (plan_key, coverage_level, tier, days_supply)",
    f"OPTIMIZE {GLD}.fact_pricing_unit_cost ZORDER BY (plan_key, ndc, days_supply)",
    f"OPTIMIZE {GLD}.fact_pharmacy_network ZORDER BY (plan_key, pharmacy_number, pharmacy_zipcode)",
    f"OPTIMIZE {GLD}.fact_formulary_drug ZORDER BY (formulary_id, ndc, rxcui)",

    # Agg tables
    f"OPTIMIZE {GLD}.agg_plan_formulary_metrics ZORDER BY (plan_key)",
    f"OPTIMIZE {GLD}.agg_plan_network_metrics ZORDER BY (plan_key)",
    f"OPTIMIZE {GLD}.agg_plan_pricing_metrics ZORDER BY (plan_key)",
]

for stmt in opt_statements:
    try:
        spark.sql(stmt)
        print("OK:", stmt)
    except Exception as e:
        print("SKIP/FAIL:", stmt, "|", str(e)[:200])

print("Gold build complete.")


In [0]:
%sql
OPTIMIZE cms_partd_gold.mart.agg_plan_insulin_cost_metrics
ZORDER BY (plan_key);

OPTIMIZE cms_partd_gold.mart.agg_plan_insulin_access_metrics
ZORDER BY (plan_key);

OPTIMIZE cms_partd_gold.mart.agg_plan_insulin_friendliness
ZORDER BY (plan_key);

In [0]:
%sql
-- Insulin tagged?
SELECT is_insulin, COUNT(*) FROM cms_partd_silver.core.slv_basic_formulary GROUP BY is_insulin;

-- Plans with insulin cost rows
SELECT COUNT(DISTINCT plan_key) FROM cms_partd_silver.core.slv_plan_insulin_cost;

-- Friendliness table coverage
SELECT COUNT(*) FROM cms_partd_gold.mart.agg_plan_insulin_friendliness;
